In [63]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [64]:
data = pd.read_csv('train.csv')
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [65]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.0

In [66]:
X_train[:, 0].shape
#checking the data shape tp match actual data

(784,)

In [67]:
#inititalizing the parameters

def init_params():
    W1 = np.random.rand(10, 784) - 0.5 #W1 is the weight matrix for the first layer of the neural network, initialized with random values between -0.5 and 0.5, and has a shape of (10, 784) where 10 is the number of neurons in the hidden layer and 784 is the number of input features (28x28 pixels).
    b1 = np.random.rand(10, 1) - 0.5 #b1 is the bias vector for the first layer, initialized with random values between -0.5 and 0.5, and has a shape of (10, 1).
    W2 = np.random.rand(10, 10) - 0.5 #W2 is the weight matrix for the second layer of the neural network, initialized with random values between -0.5 and 0.5, and has a shape of (10, 10).
    b2 = np.random.rand(10, 1) - 0.5 #b2 is the bias vector for the second layer, initialized with random values between -0.5 and 0.5, and has a shape of (10, 1).
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z) #ReLU is an activation function that returns the input value if it is greater than 0, and returns 0 otherwise. It is applied element-wise to the input array Z.

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1 #Z1 is the linear transformation of the input X using the weights W1 and biases b1 for the first layer of the neural network.
    A1 = ReLU(Z1) #A1 is the output of the first layer after applying the ReLU activation function to Z1.
    Z2 = W2.dot(A1) + b2 #Z2 is the linear transformation of A1 using the weights W2 and biases b2 for the second layer of the neural network.
    A2 = softmax(Z2) #A2 is the output of the second layer after applying the ReLU activation function to Z2.
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1)) #one_hot_Y is a matrix of zeros with a shape of (number of samples, number of classes), where the number of classes is determined by the maximum value in Y plus one.
    one_hot_Y[np.arange(Y.size), Y] = 1 #This line sets the appropriate elements in the one_hot_Y matrix to 1 based on the class labels in Y. It uses advanced indexing to assign a value of 1 to the correct position for each sample.
    one_hot_Y = one_hot_Y.T
    return one_hot_Y 

def deriv_ReLU(Z):
    return Z > 0

def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    dZ2 = A2 - one_hot(Y)
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2
    

In [68]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()
    for i in range (iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2,  db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 50 == 0:
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2
    

In [69]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 500, 0.1)

Iteration:  0
[1 1 1 ... 1 1 1] [9 9 7 ... 0 5 9]
Accuracy:  0.0773170731707317
Iteration:  50
[7 8 5 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.4598048780487805
Iteration:  100
[9 9 7 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.6427560975609756
Iteration:  150
[9 9 7 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.7068536585365853
Iteration:  200
[9 9 7 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.7448048780487805
Iteration:  250
[9 9 7 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.7741219512195122
Iteration:  300
[9 9 7 ... 0 3 7] [9 9 7 ... 0 5 9]
Accuracy:  0.7946341463414635
Iteration:  350
[9 9 7 ... 0 3 9] [9 9 7 ... 0 5 9]
Accuracy:  0.8104878048780488
Iteration:  400
[9 9 7 ... 0 3 9] [9 9 7 ... 0 5 9]
Accuracy:  0.823170731707317
Iteration:  450
[9 9 7 ... 0 3 9] [9 9 7 ... 0 5 9]
Accuracy:  0.8339756097560975
